# 🧩 GPU Partitioning Playground
### Tasar'u · NVIDIA Platform & Cert Prep — hands-on lab

Real **MIG** and **vGPU** need an A100/H100/Blackwell GPU, so you can't run true
hardware partitioning on Colab or Kaggle's T4s. But every *idea* behind partitioning
has a software stand-in you **can** run here — and doing so makes the exam concepts stick.

| Concept from the slides | Needs | What you'll run in this notebook |
|---|---|---|
| **MIG** — hardware isolation | A100/H100 | Cap a process to a slice of VRAM (`set_per_process_memory_fraction`) → prove isolation with a deliberate OOM |
| **Time-slicing** — best-effort sharing | any GPU | 1 → 2 → 4 concurrent inference clients on **one** GPU → measure the sharing tax |
| **vGPU / whole-GPU assignment** | a scheduler | `CUDA_VISIBLE_DEVICES` → give each tenant a full GPU (Kaggle 2×T4) |
| **Model split across partitions** | multi-GPU | Shard a model too big for 16 GB across **two** T4s with `device_map` |

**Platforms:** Colab (1× T4) covers sections 1–3. Kaggle (**2× T4**, Settings → Accelerator → *GPU T4 ×2*) also unlocks sections 4–5.

> Everything here is an LLM workload — the same inference you'd serve in production, just partitioned.


## 0 · Setup

In [ ]:
!pip -q install "transformers>=4.44" accelerate "torch" matplotlib 2>/dev/null
import torch, os, sys, time
print("torch", torch.__version__, "| CUDA", torch.version.cuda)
print("GPUs visible:", torch.cuda.device_count())
!nvidia-smi --query-gpu=index,name,memory.total,memory.used --format=csv

## 1 · Why no real MIG here?
`nvidia-smi -L` lists your GPUs. On an A100 with MIG enabled you'd see entries like
`MIG 1g.10gb Device 0`. On a T4 you'll just see the whole card — T4 (Turing) has **no MIG support**.
That's fine: we'll *emulate* the isolation in software.

In [ ]:
!nvidia-smi -L
# MIG status query — on T4 this reports "Not Supported", which is the point.
!nvidia-smi --query-gpu=mig.mode.current --format=csv || echo "MIG not supported on this GPU"
print("\nGPU 0:", torch.cuda.get_device_name(0),
      "| compute capability", torch.cuda.get_device_capability(0),
      "\n(MIG needs compute capability 8.0+ = Ampere/Hopper/Blackwell. T4 is 7.5.)")

## 2 · "Poor-man's MIG" — carve one GPU into capped instances
`torch.cuda.set_per_process_memory_fraction(f, device)` caps a **process** to a fraction
`f` of the GPU's memory. That's the software echo of a MIG slice: the process simply
**cannot** allocate past its share — try, and you get a clean OOM.

We run this in a *subprocess* so the cap starts on a fresh CUDA context.

In [ ]:
cap_worker = r'''
import os, torch
frac = float(os.environ.get("MEM_FRAC", "0.15"))
torch.cuda.set_per_process_memory_fraction(frac, 0)
total = torch.cuda.get_device_properties(0).total_memory / 1e9
cap_gb = frac * total
print(f"Instance capped at {frac:.0%} of {total:.1f} GB = {cap_gb:.1f} GB")
# 1) allocation that fits inside the slice -> OK
small = torch.empty(int(cap_gb*0.5*1e9)//2, dtype=torch.float16, device="cuda")
print(f"  allocated {small.numel()*2/1e9:.1f} GB inside the slice ... OK")
# 2) allocation that busts the slice -> OOM (isolation!)
try:
    big = torch.empty(int(total*0.9*1e9)//2, dtype=torch.float16, device="cuda")
    print("  unexpected: large alloc succeeded")
except RuntimeError as e:
    print("  busting the cap -> OOM (this is the isolation):", str(e).split(".")[0])
'''
open("cap_worker.py","w").write(cap_worker)
import subprocess, sys
subprocess.run([sys.executable, "cap_worker.py"])

### 2b · Four tenants sharing one GPU at the same time
Now launch **4 capped workers concurrently**, each loading a small LLM and running
inferences. They coexist on a single T4 — exactly what MIG does for multi-tenant
inference, here approximated by memory caps + the OS scheduler.

In [ ]:
tenant = r'''
import os, time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
frac = float(os.environ.get("MEM_FRAC","0.22")); name = os.environ.get("WORKER","t")
torch.cuda.set_per_process_memory_fraction(frac, 0)
mid = "sshleifer/tiny-gpt2"   # tiny + fast so 4 fit trivially; swap for "Qwen/Qwen2.5-0.5B"
tok = AutoTokenizer.from_pretrained(mid); model = AutoModelForCausalLM.from_pretrained(mid).half().cuda()
ids = tok("The AI data center", return_tensors="pt").to("cuda")
t0=time.time(); n=0
for _ in range(30):
    with torch.no_grad(): model.generate(**ids, max_new_tokens=16, do_sample=False)
    n+=1
peak = torch.cuda.max_memory_allocated(0)/1e9
print(f"[{name}] cap={frac:.0%}  {n} gens  peak {peak:.2f} GB  {n/(time.time()-t0):.1f} gen/s", flush=True)
'''
open("tenant.py","w").write(tenant)
import subprocess, sys, os
procs=[]
for i in range(4):
    env=dict(os.environ, MEM_FRAC="0.22", WORKER=f"tenant{i}")
    procs.append(subprocess.Popen([sys.executable,"tenant.py"], env=env))
for p in procs: p.wait()
print("\n→ 4 isolated tenants ran on ONE physical GPU. That's the MIG use-case, in software.")

## 3 · The time-slicing tax
When several clients share a GPU *without* isolation, the GPU **time-slices** between
them. More clients ≠ proportionally more work — they contend. We measure it directly:
load one model, then hammer it with 1, 2, and 4 concurrent threads for a few seconds each.

(PyTorch releases the GIL during CUDA calls, so threads genuinely overlap on the GPU.)

In [ ]:
import threading, time
from transformers import AutoModelForCausalLM, AutoTokenizer
mid = "sshleifer/tiny-gpt2"
tok = AutoTokenizer.from_pretrained(mid); model = AutoModelForCausalLM.from_pretrained(mid).half().cuda().eval()
ids = tok("Scaling AI infrastructure", return_tensors="pt").to("cuda")

def bench(k, secs=6):
    stop = time.time()+secs; counts=[0]*k
    def run(i):
        while time.time() < stop:
            with torch.no_grad(): model.generate(**ids, max_new_tokens=16, do_sample=False)
            counts[i]+=1
    ts=[threading.Thread(target=run,args=(i,)) for i in range(k)]
    [t.start() for t in ts]; [t.join() for t in ts]
    total=sum(counts); return total/secs, (secs*1000*k)/max(total,1)  # gen/s, ~ms/gen per client

rows=[]
for k in [1,2,4]:
    thr, lat = bench(k)
    rows.append((k, thr, lat)); print(f"{k} client(s): {thr:5.1f} gen/s total   ~{lat:6.1f} ms/gen per client")
print("\nNotice: total throughput flattens and per-client latency climbs — that's the time-slice tax.")

In [ ]:
import matplotlib.pyplot as plt
ks=[r[0] for r in rows]; thr=[r[1] for r in rows]; lat=[r[2] for r in rows]
fig,ax=plt.subplots(1,2,figsize=(10,3.4))
ax[0].plot(ks,thr,'o-',color="#4f29b7"); ax[0].set_title("Total throughput vs clients"); ax[0].set_xlabel("concurrent clients"); ax[0].set_ylabel("gen/s")
ax[1].plot(ks,lat,'o-',color="#e2426b"); ax[1].set_title("Per-client latency vs clients"); ax[1].set_xlabel("concurrent clients"); ax[1].set_ylabel("ms/gen")
plt.tight_layout(); plt.show()

### 3b · (Optional) CUDA MPS — concurrent kernels instead of time-slicing
**Multi-Process Service** lets kernels from different processes run *concurrently* on one
GPU instead of taking turns. On Colab (you have root) you can try:
```bash
nvidia-cuda-mps-control -d          # start the MPS daemon
# ...run your multi-process workload; kernels now overlap...
echo quit | nvidia-cuda-mps-control # stop it
```
It won't always be permitted on shared runtimes — that's fine, the concept is the exam point:
**time-slicing = take turns · MPS = share concurrently · MIG = hardware-isolated slices.**

## 4 · Whole-GPU assignment across two GPUs  *(Kaggle 2×T4)*
This is what a scheduler like **Run:ai** or the **K8s device plugin** does: hand each
tenant its own GPU with `CUDA_VISIBLE_DEVICES`. Skips automatically if you only have one GPU.

In [ ]:
import torch, subprocess, sys, os
if torch.cuda.device_count() < 2:
    print("Only 1 GPU visible — switch Kaggle to 'GPU T4 ×2' to run sections 4 & 5.")
else:
    w = r'''
import os, torch
print(f"  worker sees GPU: {torch.cuda.get_device_name(0)}  (CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')})")
'''
    open("assign.py","w").write(w)
    for g in ["0","1"]:
        env=dict(os.environ, CUDA_VISIBLE_DEVICES=g)
        print(f"Tenant pinned to physical GPU {g}:")
        subprocess.run([sys.executable,"assign.py"], env=env)
    print("\n→ Two independent tenants, one full GPU each. No sharing, hard isolation.")

## 5 · Split ONE model across both partitions  *(Kaggle 2×T4)*
A 7B model in fp16 is ~14 GB — tight on a single 16 GB T4. Tell 🤗 Accelerate to place
layers across **both** GPUs with a memory budget per device. `hf_device_map` then shows
which layers landed on `cuda:0` vs `cuda:1` — model parallelism you can see.

In [ ]:
import torch
if torch.cuda.device_count() < 2:
    print("Need 2 GPUs for this section (Kaggle 'GPU T4 ×2').")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    mid = "Qwen/Qwen2.5-1.5B"   # small enough to download fast; forced to split by the budget below
    tok = AutoTokenizer.from_pretrained(mid)
    model = AutoModelForCausalLM.from_pretrained(
        mid, torch_dtype=torch.float16, device_map="auto",
        max_memory={0: "1GiB", 1: "1GiB", "cpu": "8GiB"})   # tiny budget forces a cross-GPU split
    from collections import Counter
    where = Counter(str(v) for v in model.hf_device_map.values())
    print("Layer placement across devices:", dict(where))
    ids = tok("Multi-GPU inference means", return_tensors="pt").to("cuda:0")
    out = model.generate(**ids, max_new_tokens=20, do_sample=False)
    print("\nGenerated:", tok.decode(out[0], skip_special_tokens=True))
    print("\n→ One model, its layers partitioned across two GPUs — pipeline/model parallelism, live.")

## 6 · What you just proved (and how it maps to the exam)

| You ran | Real NVIDIA feature | One-line exam hook |
|---|---|---|
| Memory-fraction cap + OOM | **MIG** | Hardware-isolated slices of one GPU for multi-tenant inference |
| 4 capped tenants at once | **MIG / vGPU** | Pack many small jobs onto one big GPU |
| 1→2→4 client throughput curve | **Time-slicing** | Best-effort sharing; no isolation guarantee |
| `CUDA_VISIBLE_DEVICES` per tenant | **vGPU / Run:ai / K8s device plugin** | Whole-GPU assignment & pooling |
| `device_map` across 2×T4 | **Tensor/Pipeline parallelism** | Split a model that won't fit on one GPU |

**Reflection (write your answers):**
1. Your memory cap is *software*. Name two things a real MIG slice guarantees that a memory
   fraction does **not**. *(hint: dedicated SMs, fault isolation, predictable performance)*
2. When would you choose **MIG** over **time-slicing** for an inference service? Over **whole-GPU**?
3. In section 5, why did the first token still have to wait for both GPUs?
